In [243]:
# initialize stuff
import sys
import logging
import importlib

import numpy as np
from numpy.typing import NDArray
import matplotlib.pyplot as plt

from tqdm import tqdm
from scipy.optimize import curve_fit

# import theory stuff

HOME_DIR = "/Users/oliver/Documents/p5control-bluefors-evaluation"
sys.path.append(HOME_DIR)

import superconductivity.api as sc

from superconductivity.style.cpd4 import cmap, colors

import numpy as np

from matplotlib.markers import MarkerStyle
from matplotlib.lines import Line2D

import superconductivity.api as sc

from superconductivity.api import G_0_muS
from superconductivity.api import NDArray64

from IPython import get_ipython

_ip = get_ipython()
if _ip is not None:
    _ip.run_line_magic("matplotlib", "inline")
    _ip.run_line_magic("reload_ext", "autoreload")
    _ip.run_line_magic("autoreload", "2")

    _ip.run_line_magic(
        "config",
        "InlineBackend.print_figure_kwargs = {'bbox_inches': None, 'pad_inches': 0.0}",
    )
    _ip.run_line_magic("config", 'InlineBackend.figure_format = "retina"')  # or "png"
    _ip.run_line_magic(
        "config", "InlineBackend.rc = {'figure.dpi': 300}"
    )  # choose a value you like

In [ ]:
from superconductivity.evaluation import list_specific_keys_and_values, get_ivt

importlib.reload(sys.modules["superconductivity.evaluation"])

h5path = "data/2023-11-04_G0_antenna.hdf5"
measurement = "frequency_at_15GHz"
keys, P_out_dBm = list_specific_keys_and_values(
    h5path=h5path,
    measurement="frequency_at_15GHz",
    strip0="=",
    strip1="dBm",
)

In [ ]:
from superconductivity.evaluation import get_ivt
from superconductivity.evaluation.psd import get_psd

ivt_raw = []
ivf_raw = []


for i, key in enumerate(keys):
    _ivt_raw = get_ivt(
        h5path=h5path,
        measurement=measurement,
        specific_key=key,
        amp_voltage=10000,
        amp_current=1000,
        trigger_values=1,
        skip=5,
        time_relative=False,
    )
    _I_fft_nA, _V_fft_mV, _f_Hz = get_psd(
        I_nA=_ivt_raw[0],
        V_mV=_ivt_raw[1],
        t_s=_ivt_raw[2],
        detrend=False,
    )
    ivt_raw.append(_ivt_raw)
    ivf_raw.append((_I_fft_nA, _V_fft_mV, _f_Hz))

In [256]:
from superconductivity.visuals.plotly import get_ivt_sliders

fig_I_t, fig_V_t, fig_I_V = get_ivt_sliders(
    y_values=P_out_dBm,
    t_s_list=t_lists_s,
    I_nA_list=I_lists_nA,
    V_mV_list=V_lists_mV,
    y_label="<i>P</i> (dBm)",
    name="0 raw",  # saves htmls if set
    dataset="visuals/",
)

fig_I_t, fig_V_t, fig_I_V = get_ivt_sliders(
    y_values=P_out_dBm,
    t_s_list=f_lists_Hz,
    I_nA_list=fI_lists_nA,
    V_mV_list=fV_lists_mV,
    y_label="<i>P</i> (dBm)",
    name="1 fft",  # saves htmls if set
    dataset="visuals/",
)

<string>:29: ComplexWarning:

Casting complex values to real discards the imaginary part

<string>:28: ComplexWarning:

Casting complex values to real discards the imaginary part



In [257]:
# Build. HTML and Luanti worlds
dataset = "visuals"

from superconductivity.visuals.html import build_html

build_html(
    dataset=dataset,
    title="whole dataset",
    title_html="Atomic Contact at 15 GHz",
)

In [228]:
from superconductivity.utilities.functions import ragged_to_array

I_array_nA = ragged_to_array(I_lists_nA, fill_value=np.nan)
V_array_mV = ragged_to_array(V_lists_mV, fill_value=np.nan)
t_array_s = ragged_to_array(t_lists_s, fill_value=np.nan)

In [232]:
i = 0
plt.plot(
    V_array_mV[i, :],
    I_array_nA[i, :],
    ".",
)
plt.show()

In [124]:
from utilities.ivevaluation import IVEvaluation

importlib.reload(sys.modules["utilities.ivevaluation"])

eva = IVEvaluation()
eva.file_directory = ""
eva.file_folder = "data/"
eva.setAmplifications(10000, 1000)
eva.setV(1e-3, voltage_bins=2000)
eva.setI(180e-9, current_bins=2000)
eva.downsample_frequency = 43

eva.sub_folder = ""

eva.file_name = "2023-11-04_G0_antenna.hdf5"
eva.eva_temperature = True
eva.setA(0, 0.25, 250)

eva.title = "Amplitude Study (15GHz, Antenna)"
eva.setMeasurement("frequency_at_15GHz")
eva.setKeys(index_0=3, index_1=-3, norm=1)
eva.addKey("nu=-31.0dBm", -1000)
(eva.up_sweep_raw,) = eva.getMaps([1])
(eva.up_sweep,) = eva.getMapsAmplitude([eva.up_sweep_raw])
eva.y_axis = eva.amplitude_axis
eva.saveData()

(base) ... BaseClass initialized.
(base eva) ... BaseEvaluation initialized.
(iv eva) ... IVEvaluation initialized.
(base) Amplitude Study (15GHz, Antenna)
(iv eva) getBackupTemperature()
100%|██████████| 33/33 [00:00<00:00, 35.02it/s]
(iv eva) getMapsAmplitude()
(base) saveData()


In [125]:
from utilities.ivplot import IVPlot

importlib.reload(sys.modules["utilities.ivevaluation"])
eva = IVPlot()
eva.sub_folder = ""
eva.title = "Amplitude Study (15GHz, Antenna)"
eva.loadData()
eva.to_plot = eva.up_sweep
eva.plot_all()

(base) ... BaseClass initialized.
(base eva) ... BaseEvaluation initialized.
(iv eva) ... IVEvaluation initialized.
(base) ... BaseClass initialized.
(base plot) ... BasePlot initialized.
(iv plot) ... IVPlot initialized.
(base) Amplitude Study (15GHz, Antenna)
(base) loadData()
(base plot) saveFigure()
(base plot) saveFigure()


In [80]:
V_mV = eva.mapped["voltage_axis"] * 1e3
A_out_mV = eva.mapped["amplitude_axis"] * 1e3

I_exp_nA = eva.up_sweep["current"] * 1e9
T_exp_K = eva.up_sweep["temperature"]
dIdV_exp = eva.up_sweep["differential_conductance"]

I0exp_nA = I_exp_nA[0, :]

In [ ]:
t0_s = eva.up_sweep_raw["time_start"][0]
t1_s = eva.up_sweep_raw["time_stop"][0]
I0_nA = np.nanmin(eva.up_sweep_raw["current"][0]) * 1e9
I1_nA = np.nanmax(eva.up_sweep_raw["current"][0]) * 1e9

dIdt_nA_s = np.abs(I1_nA - I0_nA) / np.abs(t1_s - t0_s)
dIdt_nA_s = 23.11

In [203]:
tau = np.array([0.78, 0.66, 0.31, 0.16, 0.08, 0.03])
tau = np.array([0.78, 0.78, 0.66])
G_N = np.sum(tau)

Delta_meV = 0.1885
gamma_meV = 0
A_mV = 0.0
nu_GHz = 10.0

Isw = 0.045
T_K = 0.1
C_pF = 1.08e-3

# V_ = np.linspace(-0.1, 0.1, 2001)
# V_mV = V_ * Delta_meV

from numpy import poly
from scipy.signal import savgol_filter

from superconductivity.models.ha_sym import get_I_ha_sym_nA as get_I_ha_nA
from superconductivity.models.abs import get_Ic_abs_nA

Iqp_nA = np.zeros_like(V_mV)
Ic_nA = 0.0
for i_tau, tau_i in enumerate(tqdm(tau)):
    Iqp_nA += get_I_ha_nA(
        V_mV=V_mV, tau=tau_i, T_K=T_K, Delta_meV=Delta_meV, gamma_meV=gamma_meV
    )
    Ic_nA += get_Ic_abs_nA(Delta_meV=Delta_meV, tau=tau_i, T_K=T_K)

Isw_nA = Isw * Ic_nA

Iqp_nA = savgol_filter(Iqp_nA, window_length=200, polyorder=2)

100%|██████████| 3/3 [00:00<00:00, 552.42it/s]


In [204]:
from superconductivity.models.rcsj import get_I_rcsj_nA

importlib.reload(sys.modules["superconductivity.models.rcsj"])

Ircsj_nA = get_I_rcsj_nA(
    V_mV=V_mV,
    I_qp_nA=Iqp_nA,
    I_sw_nA=Isw_nA,
    T_K=T_K,
    GN_G0=G_N,
    C_pF=C_pF,
    A_mV=A_mV,
    nu_GHz=nu_GHz,
    model="rsj",
    include_shunt=False,
    n_periods=10,
)

In [205]:
from superconductivity.models.rcsj import get_I_rstj_slow_nA

I_nA = get_I_rstj_slow_nA(
    V_mV=V_mV,
    I_qp_nA=Iqp_nA,
    I_sw_nA=Ic_nA,
    T_eff_K=0.11,
    GN_G0=G_N,
    ramp_rate_nA_per_s=dIdt_nA_s,
    sigma_I_nA=20.0,
    include_shunt=False,
)

In [206]:
_ip.run_line_magic("matplotlib", "qt")

GN_muS = G_N * sc.G_0_muS
dG0exp_GN = np.gradient(I0exp_nA, V_mV) / (GN_muS)
dGqp_GN = np.gradient(Iqp_nA, V_mV) / (GN_muS)
dGrcsj_GN = np.gradient(Ircsj_nA, V_mV) / (GN_muS)
dG_GN = np.gradient(I_nA, V_mV) / (GN_muS)

plt.close("all")
fig, [axI, axG] = plt.subplots(nrows=2, sharex=True)

axI.plot(V_mV, I0exp_nA, "k.", label="$I_{exp}$")
axI.plot(V_mV, Iqp_nA, label="$I_{qp}$")
axI.vlines(0, ymin=-Isw_nA, ymax=Isw_nA, label="$I_{sw}$")
axI.plot(V_mV, Ircsj_nA, "r.-", label="$I_{rcsj}$")
# axI.plot(V_mV, I_nA, ".-", label="$I_{rstj_slow}$")
axI.legend()

axG.plot(V_mV, dG0exp_GN, "k.", label="$dG_{exp}$")
axG.plot(V_mV, dGqp_GN, label="$dG_{qp}$")
axG.plot(V_mV, dGrcsj_GN, "r.-", label="$dG_{rcsj}$")
# axG.plot(V_mV, dG_GN, ".-", label="$dG_{rstj_slow}$")
axG.legend()

In [70]:
Ic_nA

np.float64(56.90913900323461)

np.float64(23.116739136050708)